# Data collection

Imports

In [19]:
import pandas as pd
from pathlib import Path
import yfinance as yf

### Step 1: Convert historical membership data to historical tickers dataframe & store in one file.

In [20]:
data_folder_path = Path("data")
membership_snapshots_path = data_folder_path / "membership_snapshots"

# Create the folder and its parents if they don't exist
membership_snapshots_path.mkdir(parents=True, exist_ok=True)

print(membership_snapshots_path.exists())

True


In [21]:
series_list = []

# The files have no extension
for file_path in sorted(membership_snapshots_path.glob("*")):
    s = pd.read_csv(
        file_path,
        sep=r"\t+",
        engine="python",
        header=None,
        usecols=[0]
    )[0]

    s = s.str.strip().str.replace('/', '-', regex=False) # fix ticker formatting to match yfinance format

    s.name = file_path.stem[3:] # remove the "spy" prefix, leaving the name as YYYYMM
    series_list.append(s)

membership_df = pd.concat(series_list, axis=1)

In [22]:
print(membership_df.head())

  201601 201602 201603 201604 201605 201606 201607 201608 201609 201610  ...  \
0      A      A      A      A      A      A      A      A      A      A  ...   
1     AA     AA     AA     AA     AA     AA     AA     AA     AA     AA  ...   
2    AAL    AAL    AAL    AAL    AAL    AAL    AAL    AAL    AAL    AAL  ...   
3    AAP    AAP    AAP    AAP    AAP    AAP    AAP    AAP    AAP    AAP  ...   
4   AAPL   AAPL   AAPL   AAPL   AAPL   AAPL   AAPL   AAPL   AAPL   AAPL  ...   

  202507 202508 202509 202510 202511 202512 202601 202602 202603 202604  
0      A      A      A      A      A      A      A      A      A      A  
1   AAPL   AAPL   AAPL   AAPL   AAPL   AAPL   AAPL   AAPL   AAPL   AAPL  
2   ABBV   ABBV   ABBV   ABBV   ABBV   ABBV   ABBV   ABBV   ABBV   ABBV  
3   ABNB   ABNB   ABNB   ABNB   ABNB   ABNB   ABNB   ABNB   ABNB   ABNB  
4    ABT    ABT    ABT    ABT    ABT    ABT    ABT    ABT    ABT    ABT  

[5 rows x 124 columns]


Save to CSV

In [23]:
membership_df.to_csv(data_folder_path / "membership_df.csv", index=False)

### Step 2: Store historical OHLCV price data

Get all unique tickers from January 2016 to January 2026

In [24]:
unique_tickers = pd.Series(membership_df.values.ravel()).dropna().unique().tolist()
print(unique_tickers)

['A', 'AA', 'AAL', 'AAPL', 'AAP', 'ABBV', 'ABNB', 'ABT', 'ABMD', 'ACGL', 'ACN', 'ADBE', 'ADI', 'ADM', 'ADP', 'ADS', 'ADSK', 'AEE', 'AEP', 'ADT', 'AES', 'AFL', 'AET', 'AIG', 'AIZ', 'AGN', 'AIV', 'AJG', 'AKAM', 'ALB', 'ALGN', 'ALK', 'ALL', 'ALLE', 'AMAT', 'AMCR', 'ALXN', 'AMD', 'AME', 'AMGN', 'AMP', 'AMG', 'AMT', 'AMTM', 'AMZN', 'ANET', 'ANSS', 'AON', 'AOS', 'AN', 'APA', 'ANDV', 'APD', 'APH', 'APO', 'APC', 'APTV', 'APP', 'ARE', 'ATO', 'AVB', 'ARES', 'ARG', 'ATVI', 'AVGO', 'ARNC', 'AVY', 'AWK', 'AXON', 'AXP', 'AZO', 'BA', 'AYI', 'BAC', 'BALL', 'BAX', 'BBY', 'BBBY', 'BBWI', 'BDX', 'BEN', 'BF-B', 'BCR', 'BG', 'BIIB', 'BK', 'BKNG', 'BHI', 'BIO', 'BKR', 'BLDR', 'BLK', 'BMY', 'BHF', 'BR', 'BRK-B', 'BRCM', 'BRO', 'BSX', 'BWA', 'BX', 'BXLT', 'BXP', 'C', 'CAG', 'CA', 'CAH', 'CARR', 'CAT', 'CAM', 'CB', 'CBOE', 'CBRE', 'CCI', 'CCE', 'CCL', 'CDNS', 'CELG', 'CDW', 'CERN', 'CEG', 'CF', 'CE', 'CFG', 'CHD', 'CHK', 'CHRW', 'CHTR', 'CI', 'CINF', 'CL', 'CLX', 'CIEN', 'CMA', 'CMCSA', 'CME', 'CMG', 'CMI', 'C

Pull data for these tickers from January 2014 to January 2026 (2 extra years as a loopback period)

In [25]:
start_date = "2014-01-01"
end_date = "2026-01-03" # first trading day in 2026; yfinance does not include the end date

In [26]:
historical_ohlcv_df = yf.download(unique_tickers, start=start_date, end=end_date, group_by='ticker', auto_adjust=True) # auto adjust=True to account for splits and dividends

[                       0%                       ]$LLL: possibly delisted; no timezone found
[                       0%                       ]  2 of 705 completed$GMCR: possibly delisted; no timezone found
$MRO: possibly delisted; no timezone found
[                       0%                       ]  3 of 705 completed$ETFC: possibly delisted; no timezone found
[                       1%                       ]  5 of 705 completed$DO: possibly delisted; no timezone found
[                       1%                       ]  6 of 705 completed$RTN: possibly delisted; no timezone found
$GGP: possibly delisted; no timezone found
[                       1%                       ]  8 of 705 completed$NFX: Data doesn't exist for startDate = 1388552400, endDate = 1767416400
[                       1%                       ]  9 of 705 completed$LEG: Data doesn't exist for startDate = 1388552400, endDate = 1767416400
[                       1%                       ]  10 of 705 completed$PXD: pos

In [30]:
print(historical_ohlcv_df.head())

Ticker      PCL                        LLL                           ...  \
Price      Open High Low Close Volume Open High Low Close Adj Close  ...   
Date                                                                 ...   
2014-01-02  NaN  NaN NaN   NaN    NaN  NaN  NaN NaN   NaN       NaN  ...   
2014-01-03  NaN  NaN NaN   NaN    NaN  NaN  NaN NaN   NaN       NaN  ...   
2014-01-06  NaN  NaN NaN   NaN    NaN  NaN  NaN NaN   NaN       NaN  ...   
2014-01-07  NaN  NaN NaN   NaN    NaN  NaN  NaN NaN   NaN       NaN  ...   
2014-01-08  NaN  NaN NaN   NaN    NaN  NaN  NaN NaN   NaN       NaN  ...   

Ticker           ACGL                                                 ROST  \
Price            Open       High        Low      Close   Volume       Open   
Date                                                                         
2014-01-02  18.783414  18.875336  18.162162  18.184349  2592900  32.832875   
2014-01-03  18.152653  18.219215  17.753275  17.835686  3620400  32.819630   
2

Save to parquet file (good for handling MultiIndex colums)

In [31]:
historical_ohlcv_df.to_parquet(
    data_folder_path / "historical_ohlcv_df.parquet"
)

In [33]:
historical_ohlcv_df.index[-1]

Timestamp('2026-01-02 00:00:00')